# Gradient Cancellation Proof
Proves that an antipodal ReLU pair has zero combined gradient under a symmetric distribution and even target.

In [ ]:
import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

torch.manual_seed(42)
np.random.seed(42)
matplotlib.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False})

# ── Data ──────────────────────────────────────────────────────────────────────────────
def sample_uniform(n: int = 4096) -> torch.Tensor:
    """x ~ Uniform(-1, 1), shape (n,)."""
    return torch.empty(n).uniform_(-1, 1)

# ── Target y*(x) = |x| + relu(|x|-0.3) + relu(|x|-0.7) ────────────────────────────────────
def target(x: torch.Tensor) -> torch.Tensor:
    ax = x.abs()
    return ax + torch.relu(ax - 0.3) + torch.relu(ax - 0.7)

# ── Fixed-neuron contribution (pairs 2 & 3 from target table) ─────────────────────────────
_W_FIX = torch.tensor([ 1., -1.,  1., -1.])
_B_FIX = torch.tensor([-0.3, -0.3, -0.7, -0.7])
_V_FIX = torch.tensor([ 1.,  1.,  1.,  1.])

def f_fixed(x: torch.Tensor) -> torch.Tensor:
    """Contribution of the 4 fixed optimal neurons (frozen weights, no learnable params)."""
    pre = x.unsqueeze(1) * _W_FIX + _B_FIX   # (N, 4)
    return (torch.relu(pre) * _V_FIX).sum(1)   # (N,)

# ── Gradient norm helper ─────────────────────────────────────────────────────────────
def grad_norm(*tensors) -> float:
    return sum((t.grad.norm().item() for t in tensors if t.grad is not None), 0.0)

print("Helpers ready.")

## Section 1 — Target function

In [ ]:
x_plot = torch.linspace(-1, 1, 400)
y_plot = target(x_plot)

# Kink locations
kinks = [-0.7, -0.3, 0.0, 0.3, 0.7]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(x_plot.numpy(), y_plot.numpy(), lw=2, label=r"$y^*(x)$")
for k in kinks:
    ax.axvline(k, color="gray", ls="--", lw=0.8, alpha=0.6)
ax.scatter(kinks, target(torch.tensor(kinks)).numpy(),
           zorder=5, color="tab:orange", label="kinks")
ax.set_xlabel("x")
ax.set_title(r"$y^*(x) = |x| + \mathrm{relu}(|x|-0.3) + \mathrm{relu}(|x|-0.7)$  — 5 kinks, even function")
ax.legend()
plt.tight_layout()
plt.savefig("fig_target.png", dpi=150)
plt.show()
print(f"y*(0.5)  = {target(torch.tensor(0.5)):.4f}")
print(f"y*(-0.5) = {target(torch.tensor(-0.5)):.4f}  (must match — even function)")

## Part A — Algebraic proof + autograd verification

### Derivation

For the antipodal pair $(w_+, w_- = -w_+)$ with $b_+=b_-=0$, $v_+=v_-=1$, the MSE loss is

$$L = \mathbb{E}_x\!\left[(f(x) - y^*(x))^2\right]$$

The gradient w.r.t. $w_+$:

$$\frac{\partial L}{\partial w_+}
= \mathbb{E}\!\left[2(f-y^*)\cdot\mathbf{1}[w_+ x>0]\cdot x\right]$$

The gradient w.r.t. $w_-$ (chain rule, $w_- = -a$):

$$\frac{\partial L}{\partial w_-}
= \mathbb{E}\!\left[2(f-y^*)\cdot\mathbf{1}[w_- x>0]\cdot x\right]
= \mathbb{E}\!\left[2(f-y^*)\cdot\mathbf{1}[-w_+ x>0]\cdot x\right]$$

Since $\mathbf{1}[w_+x>0]$ and $\mathbf{1}[-w_+x>0]$ are complementary indicators (one is 1 whenever the other is 0, a.s.), their sum equals 1. Therefore:

$$\frac{\partial L}{\partial w_+} + \frac{\partial L}{\partial w_-}
= \mathbb{E}\!\left[2(f(x)-y^*(x))\cdot x\right]$$

**Key observation:** The pair's output is $v(\mathrm{relu}(w_+x)+\mathrm{relu}(-w_+x)) = v\,w_+\,|x|$ — an **even** function. The 4 fixed neurons also produce an even function (symmetric pairs). So $f(x)$ is even. The target $y^*(x)$ is even. Therefore $f(x)-y^*(x)$ is even, i.e. $g(-x)=g(x)$.

Under **any** zero-mean symmetric distribution $p(x)=p(-x)$:

$$\mathbb{E}[g(x)\cdot x] = \int g(x)\,x\,p(x)\,dx = 0$$

because the integrand $g(x)\cdot x$ is an **odd** function (even × odd = odd) and integrates to zero over any symmetric domain.

**Conclusion:** $\nabla_{w_+}L + \nabla_{w_-}L = 0$ exactly, for any value of $a$, any symmetric distribution, and any even target.

In [ ]:
# Part A — numerical verification of ∂L/∂w+ + ∂L/∂w- = 0
# Use a perfectly symmetric sample + float64 to achieve machine-zero gradient sum.
x_half = torch.linspace(0.001, 1, 500_000)          # 500k points on (0,1]
x_big  = torch.cat([-x_half.flip(0), x_half])       # mirror → exactly symmetric
# Cast to float64 so that sums cancel at double precision (~1e-16)
x_big  = x_big.double()
y_big  = target(x_big)

# float64 versions of fixed-neuron weights
_W_FIX64 = _W_FIX.double()
_B_FIX64 = _B_FIX.double()
_V_FIX64 = _V_FIX.double()

def f_fixed64(x):
    pre = x.unsqueeze(1) * _W_FIX64 + _B_FIX64
    return (torch.relu(pre) * _V_FIX64).sum(1)

print(f"{'a':>6}  {'∂L/∂w+':>14}  {'∂L/∂w-':>14}  {'|sum|':>14}")
print("-" * 56)

a_sweep = [0.1, 0.3, 0.5, 0.7, 0.9]
sweep_results = []   # saved for the plot in Task 5

for a in a_sweep:
    w_pos = torch.tensor( a, dtype=torch.float64, requires_grad=True)
    w_neg = torch.tensor(-a, dtype=torch.float64, requires_grad=True)

    # Forward: antipodal pair + 4 fixed optimal neurons
    f_pair = torch.relu(w_pos * x_big) + torch.relu(w_neg * x_big)  # (N,)
    f_all  = f_pair + f_fixed64(x_big)                               # (N,)
    loss   = ((f_all - y_big) ** 2).mean()

    gp, gn = torch.autograd.grad(loss, [w_pos, w_neg])
    gsum = (gp + gn).abs().item()
    sweep_results.append((a, gp.item(), gn.item(), gsum))

    print(f"{a:>6.1f}  {gp.item():>14.8f}  {gn.item():>14.8f}  {gsum:>14.2e}")

print()
all_pass = all(r[3] < 1e-12 for r in sweep_results)
print(f"All |sum| < 1e-12: {'✓ PASS' if all_pass else '✗ FAIL'}")
assert all_pass, "Gradient sum is not machine zero — check implementation."

In [ ]:
# Three-panel plot:
# (left)  ∂L/∂w+ and ∂L/∂w- vs a  (mirror images)
# (mid)   (f(x)-y*(x)) vs x        (even function)
# (right) (f(x)-y*(x))·x vs x      (odd function, expectation = 0)

a_vals = [r[0] for r in sweep_results]
gp_vals = [r[1] for r in sweep_results]
gn_vals = [r[2] for r in sweep_results]

# Use a=0.5 for the function plots
a_demo = 0.5
x_p = torch.linspace(-1, 1, 600)
f_demo = (torch.relu(a_demo * x_p) + torch.relu(-a_demo * x_p)
          + f_fixed(x_p)).detach()
err    = f_demo - target(x_p)
err_x  = err * x_p

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Left
axes[0].plot(a_vals, gp_vals, "o-", label=r"$\partial L/\partial w_+$")
axes[0].plot(a_vals, gn_vals, "s--", label=r"$\partial L/\partial w_-$")
axes[0].axhline(0, color="k", lw=0.8, ls=":")
axes[0].set_xlabel("a")
axes[0].set_title("Gradients are exact mirrors")
axes[0].legend()

# Mid
axes[1].plot(x_p.numpy(), err.numpy(), lw=2, color="tab:orange")
axes[1].axhline(0, color="k", lw=0.8, ls=":")
axes[1].set_xlabel("x")
axes[1].set_title(r"$f(x)-y^*(x)$ — even function  $[a=0.5]$")

# Right
axes[2].plot(x_p.numpy(), err_x.numpy(), lw=2, color="tab:green")
axes[2].axhline(0, color="k", lw=0.8, ls=":")
axes[2].axhline(err_x.mean().item(), color="tab:red", ls="--",
                label=f"mean = {err_x.mean().item():.2e}")
axes[2].set_xlabel("x")
axes[2].set_title(r"$(f(x)-y^*(x))\cdot x$ — odd → $\mathbb{E}[\cdot]=0$")
axes[2].legend(fontsize=10)

plt.tight_layout()
plt.savefig("fig_partA.png", dpi=150)
plt.show()

## Part B — Frozen pair + capacity argument

In [ ]:
# Part B network (filled in Task 6)

In [ ]:
# Part B training (filled in Task 7)

In [ ]:
# Part B counterfactual (filled in Task 8)

In [ ]:
# Part B plot (filled in Task 9)

## Part C — Full LoP replay

In [ ]:
# Part C loading (filled in Task 10)

In [ ]:
# Part C trap (filled in Task 11)

In [ ]:
# Part C plot (filled in Task 12)